In [1]:
from FlagEmbedding import LayerWiseFlagLLMReranker
reranker = LayerWiseFlagLLMReranker('BAAI/bge-reranker-v2-minicpm-layerwise', use_fp16=True)

C:\Users\kwu\AppData\Roaming\Python\Python39\site-packages\transformers\utils\generic.py:482: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\kwu\AppData\Roaming\Python\Python39\site-packages\transformers\utils\generic.py:339: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\kwu\AppData\Roaming\Python\Python39\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kwu\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-minicpm-layerwise. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling ba

tokenizer.model:   0%|          | 0.00/1.99M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00003-of-00003.safetensors:   0%|          | 0.00/935M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

In [2]:
import json
import os



In [4]:
with open('ground_summary/claude_test_similarity.json', 'r') as f:
    ground_summary = json.load(f)

In [6]:
g = ground_summary[0]['ground_summary']
l =ground_summary[0]['llm_summary']

In [7]:
g

'This figure illustrates a GPU-accelerated holistic instance segmentation and tracking framework using pixel embedding...'

In [8]:
l

"This framework processes a sequence of video frames, feeding each into a Deep Recurrent Network (RSHN) that leverages recurrent flow to ensure temporal continuity. The RSHN's output is then converted into pixel-wise embeddings, which are subsequently clustered by a GPU-accelerated Faster Mean-shift algorithm for speed. This efficient pipeline culminates in the generation of instance masks with tracking IDs, enabling holistic real-time instance segmentation and tracking."

In [12]:
score = reranker.compute_score([g, l], cutoff_layers=[28],normalize=True)

100%|██████████| 1/1 [00:02<00:00,  2.25s/it]


In [13]:
score

[0.8956191370492905]

In [19]:
def get_similarity(model):
    similarity = []
    with open(f'ground_summary/{model}_test_similarity.json', 'r') as f:
        summary = json.load(f)
        for s in summary:
            g = s['ground_summary']
            l = s['llm_summary']
            score = reranker.compute_score([g, l], cutoff_layers=[28],normalize=True)
            similarity.append(score[0])
    return similarity



In [21]:
model = {'claude': 0,'gpt4o':0,'gpt5':0}
for m in list(model.keys()):
    model[m] = sum(get_similarity(m))/len(get_similarity(m))



100%|██████████| 1/1 [00:02<00:00,  2.79s/it]


#### display similarity score

In [24]:
model

{'claude': 0.24001242259787675,
 'gpt4o': 0.03181633674390437,
 'gpt5': 0.21882665709656018}